# Reach heat-budget fit — executable paper trail

Reproduces the Firehole reach fit end-to-end from the **frozen** obs
bundle: grid fit → least-squares polish → null comparison (AIC/BIC) →
skill scores → ridge-ensemble bands → figures. No network: (re-)fetching
is `tools/fetch_firehole_obs.py`'s job (manual); this notebook starts
where the frozen bundle starts. Unit tests are synthetic/offline only;
this notebook runs on measured data and labels everything it finds
`fitted_not_validated_single_window`.

- Bundle: `$OBS` (default `derived/firehole_obs.json`).
  Provenance (URLs, fetched_at_utc) lives inside the bundle.
- Grid below is **coarse on purpose** (27 points + ~20 polish evals,
  a minute or two) so the trail stays re-runnable; the production
  V300k grid is 7×7×5. Conclusions must survive the coarse grid to
  be trusted — the ridge does.
- Geometry is authored (V = 300000 m³, A = 600000 m² = 2×V):
  fitted T_lat is conditional, not a measurement.
- Figures land in `$FIG_DIR` (default `figs/notebook`).
- Re-execute to check for rot:
  `jupyter nbconvert --to notebook --execute notebooks/reach_fit.ipynb`
  (requires jupyter), or stdlib-only
  `python3 tools/execute_notebook.py notebooks/reach_fit.ipynb`.


In [ ]:
import json
import os
import sys
from pathlib import Path

# Repo-root bootstrap: works whether executed from the root
# (nbconvert) or via tools/execute_notebook.py (script dir on sys.path).
ROOT = None
for cand in (Path.cwd(), Path.cwd().parent):
    if (cand / "yellowstone_python" / "__init__.py").exists():
        ROOT = cand
        break
assert ROOT is not None, "repo root not found above cwd"
sys.path.insert(0, str(ROOT))

from yellowstone_python.replay import calibrate, figures, uncertainty
from yellowstone_python.replay.hydraulics import (
    residual_diagnostics,
    skill_scores,
)

OBS = Path(os.environ.get(
    "OBS", str(ROOT / "derived" / "firehole_obs.json")))
FIG_DIR = Path(os.environ.get("FIG_DIR", str(ROOT / "figs" / "notebook")))
VOLUME_M3 = 300000.0
AREA_M2 = 600000.0  # authored 2xV (0.5 m effective depth)
Q_GRID = [2.5, 3.5, 4.5]
T_GRID = [20.0, 30.0, 40.0]
C_GRID = [0.5, 1.0, 2.0]

bundle = json.loads(OBS.read_text(encoding="utf-8"))
assert bundle.get("schema") == "yellowstone.firehole_obs.v1", "wrong bundle"
print("bundle:", OBS)
print("fetched_at_utc:", bundle.get("fetched_at_utc"))
print("schema: yellowstone.firehole_obs.v1")


In [ ]:
fit = calibrate.reach_heat_fit(
    bundle, VOLUME_M3, AREA_M2, Q_GRID, T_GRID, C_GRID, return_all=True)
champ = fit["champion"]
print(f"champion: Q={champ['q_lat']} T={champ['t_lat']} c={champ['c_mult']}")
print(f"  train RMSE {champ['train_rmse']:.3f} "
      f"holdout RMSE {champ['holdout_rmse']:.3f}")
for null in fit["null_models"]:
    print(f"  null {null['name']}: train {null['train_rmse']:.3f} "
          f"holdout {null['holdout_rmse']:.3f}")


In [ ]:
polished = calibrate.least_squares_fit(
    bundle, VOLUME_M3, AREA_M2,
    (champ["q_lat"], champ["t_lat"], champ["c_mult"]))
print(f"polished: Q={polished['q_lat']:.3f} T={polished['t_lat']:.3f} "
      f"c={polished['c_mult']:.3f} ({polished['nfev']} evals)")
print(f"  train RMSE {polished['train_rmse']:.3f} "
      f"holdout RMSE {polished['holdout_rmse']:.3f}")

fwd = calibrate.run_fit_forward(
    bundle, VOLUME_M3, AREA_M2, polished["q_lat"], polished["t_lat"],
    polished["c_mult"])
grid, modeled = fwd["grid"], fwd["modeled_c"]
target = calibrate.interp_series(
    bundle["sites"]["06036905"]["series"]["temperature_c"], grid)
upstream = calibrate.interp_series(
    bundle["sites"]["06036805"]["series"]["temperature_c"], grid)
start = next(i for i, g in enumerate(grid) if g - grid[0] >= 86400.0)
split = start + int((len(grid) - start) * 0.7)
train, holdout = slice(start, split), slice(split, len(grid))
n_train = split - start

models = [{"name": "lumped_fit_k3",
           **calibrate.aic_bic(sum((m - t) ** 2 for m, t in
                                   zip(modeled[train], target[train])),
                               n_train, 3),
           "train_rmse": polished["train_rmse"],
           "holdout_rmse": polished["holdout_rmse"]}]
for null, k in zip(fit["null_models"], (0, 2, 0)):
    models.append({"name": null["name"],
                   **calibrate.aic_bic(null["train_rmse"] ** 2 * n_train,
                                       n_train, k),
                   "train_rmse": null["train_rmse"],
                   "holdout_rmse": null["holdout_rmse"]})
for row in sorted(models, key=lambda r: r["aic"]):
    print(f"  {row['name']}: AIC {row['aic']:.1f} BIC {row['bic']:.1f}")


In [ ]:
for name, sl in (("train", train), ("holdout", holdout)):
    skill = skill_scores(modeled[sl], target[sl])
    print(f"{name}: NSE {skill['nse']:.3f} KGE {skill['kge']:.3f} "
          f"(r={skill['correlation']:.3f} "
          f"alpha={skill['alpha']:.3f} beta={skill['beta']:.3f})")
diag = residual_diagnostics(modeled[train], target[train])
print("train residual lag-1..3:",
      [round(diag["autocorr"][lag], 3) for lag in (1, 2, 3)])
print("train max Q-Q gap:", round(diag["max_qq_gap"], 3))


In [ ]:
members = uncertainty.ridge_members(
    fit["all_results"], champ["train_rmse"])
fallback = any(m["fallback_member"] for m in members)
print(f"ridge members: {len(members)} (fallback={fallback})")
if fallback:
    print("note: fallback top-5 on this coarse grid spans distant fits, "
          "so bands are diffuse and cover easily; on the production "
          "grid the tight ridge under-covers (A1 artifact).")
for m in members:
    print(f"  Q={m['q_lat']} T={m['t_lat']} c={m['c_mult']} "
          f"train {m['train_rmse']:.3f}")
trajs = uncertainty.member_trajectories(
    bundle, VOLUME_M3, AREA_M2, members)
bands = uncertainty.ensemble_bands(trajs["trajectories"])
cov = uncertainty.band_coverage(
    bands["lo"][holdout], bands["hi"][holdout], target[holdout])
print(f"holdout coverage: {cov['coverage']:.2f} at nominal 0.90 "
      f"(n={cov['n']}, band width {cov['mean_width_c']:.2f} C)")


In [ ]:
FIG_DIR.mkdir(parents=True, exist_ok=True)
p1 = figures.fit_holdout_figure(
    grid, modeled, target, upstream, train, holdout,
    "Reach heat-budget fit vs downstream gauge",
    FIG_DIR / "fit_holdout.png")
level = 0.90
p2 = figures.bands_figure(
    grid, bands["lo"], bands["mid"], bands["hi"], target[holdout],
    holdout, level, cov["coverage"], "Ridge-ensemble bands",
    FIG_DIR / "bands.png")
p3 = figures.ridge_figure(fit["all_results"], champ, polished,
                          FIG_DIR / "ridge.png")
for p in (p1, p2, p3):
    print(f"Wrote {p} ({p.stat().st_size} bytes)")


## Reading this trail

- The fit must beat both nulls on **holdout**, not just train, and win
  on AIC/BIC — otherwise the extra parameters are decoration.
- The polish should stay near the champion; a long walk along the
  ridge (Q down / T up together) confirms the split is unidentified
  and fitted T_lat is conditional on authored geometry.
- Bands describe fit ambiguity. If holdout coverage misses nominal
  badly, the error is structural bias (check residual lag-1), not
  parametric spread — do not widen the ridge rule to hide it.
- Everything above is `fitted_not_validated_single_window` until a
  second independent window confirms it.
